In [1]:
import argparse
import os
import pathlib
from datetime import datetime

import numpy as np
import pandas as pd
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [2]:
# This script never accepted a --image_based_profiles_subparent_name argument
# at all -- it hardcoded "image_based_profiles" in every path below, so it
# could never see per-patient output written under a different subparent
# name (e.g. this project's own "image_based_profiles_production_zedprofiler").
# Default preserves the previous hardcoded behavior for existing callers.
if not in_notebook:
    _arg_parser = argparse.ArgumentParser()
    _arg_parser.add_argument(
        "--image_based_profiles_subparent_name", default="image_based_profiles"
    )
    _arg_parser.add_argument("--output_subdir", default="all_patient_profiles")
    _args = _arg_parser.parse_args()
    image_based_profiles_subparent_name = _args.image_based_profiles_subparent_name
    output_subdir = _args.output_subdir
else:
    image_based_profiles_subparent_name = "image_based_profiles"
    output_subdir = "all_patient_profiles"

threshold = 1e3

In [3]:
patient_ids_file_path = pathlib.Path(f"{root_dir}/data/patient_IDs.txt").resolve(
    strict=True
)
patient_ids = pd.read_csv(patient_ids_file_path, header=None).iloc[:, 0].tolist()

In [4]:
# set up log file
log_dir = pathlib.Path("../logs/profile_validation").resolve()
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / f"profile_check_{datetime.now():%Y%m%d_%H}.log"


def log(msg, log_file):
    """Print to stdout and write to the log file."""
    print(msg)
    log_file.write(msg + "\n")


with open(log_path, "w") as log_file:
    for patient in patient_ids:
        # construct the profile path dict for this patient. Required (always
        # produced) hand-crafted entries use resolve(strict=True) so a genuine
        # gap still fails loudly; the deep-learning entries are only included
        # if the file actually exists, since a dataset with no deep-learning
        # features (e.g. ZEDProfiler-only) never produces any of them.
        profile_path_dict = {
            "sc_normalized": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/5.normalized_profiles/sc_norm.parquet"
            ).resolve(strict=True),
            "organoid_normalized": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/5.normalized_profiles/organoid_norm.parquet"
            ).resolve(strict=True),
            "sc_fs": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/6.feature_selected_profiles/sc_fs.parquet"
            ).resolve(strict=True),
            "organoid_fs": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/6.feature_selected_profiles/organoid_fs.parquet"
            ).resolve(strict=True),
            "sc_agg_well_from_norm": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/7.aggregated_profiles/sc_agg_from_norm.parquet"
            ).resolve(strict=True),
            "organoid_agg_well_from_norm": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/7.aggregated_profiles/organoid_agg_from_norm.parquet"
            ).resolve(strict=True),
            "sc_agg_well_from_fs": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/7.aggregated_profiles/sc_agg_from_fs.parquet"
            ).resolve(strict=True),
            "organoid_agg_well_from_fs": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/7.aggregated_profiles/organoid_agg_from_fs.parquet"
            ).resolve(strict=True),
            "sc_agg_treatment_from_norm": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/8.consensus_profiles/sc_consensus_from_norm.parquet"
            ).resolve(strict=True),
            "organoid_agg_treatment_from_norm": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/8.consensus_profiles/organoid_consensus_from_norm.parquet"
            ).resolve(strict=True),
            "sc_agg_treatment_from_fs": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/8.consensus_profiles/sc_consensus_from_fs.parquet"
            ).resolve(strict=True),
            "organoid_agg_treatment_from_fs": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/8.consensus_profiles/organoid_consensus_from_fs.parquet"
            ).resolve(strict=True),
        }
        optional_dl_paths = {
            "sc_sammed_normalized": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/5.normalized_profiles/sammed_sc_norm.parquet",
            "organoid_sammed_normalized": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/5.normalized_profiles/sammed_organoid_norm.parquet",
            "nucleocentric_sammed_normalized": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/5.normalized_profiles/sammed_nucleocentric_norm.parquet",
            "nucleocentric_morphem_normalized": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/5.normalized_profiles/nucleocentric_morphem_norm.parquet",
            "sc_sammed_fs": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/6.feature_selected_profiles/sammed_sc_fs.parquet",
            "organoid_sammed_fs": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/6.feature_selected_profiles/sammed_organoid_fs.parquet",
            "nucleocentric_sammed_fs": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/6.feature_selected_profiles/sammed_nucleocentric_fs.parquet",
            "nucleocentric_morphem_fs": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/6.feature_selected_profiles/nucleocentric_morphem_fs.parquet",
            "sc_sammed_agg_well": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/7.aggregated_profiles/sammed_sc_agg_well_level.parquet",
            "organoid_sammed_agg_well": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/7.aggregated_profiles/sammed_organoid_agg_well_level.parquet",
            "nucleocentric_sammed_agg_well": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/7.aggregated_profiles/sammed_nucleocentric_agg_well_level.parquet",
            "nucleocentric_morphem_agg_well": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/7.aggregated_profiles/nucleocentric_morphem_agg_well_level.parquet",
            "sc_sammed_agg_treatment": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/8.consensus_profiles/sammed_sc_consensus.parquet",
            "organoid_sammed_agg_treatment": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/8.consensus_profiles/sammed_organoid_consensus.parquet",
            "nucleocentric_sammed_agg_treatment": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/8.consensus_profiles/sammed_nucleocentric_consensus.parquet",
            "nucleocentric_morphem_agg_treatment": f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/8.consensus_profiles/nucleocentric_morphem_consensus.parquet",
        }

        log(f"\n{'=' * 60}", log_file)
        log(f"Processing patient: {patient}", log_file)
        log(f"{'=' * 60}", log_file)

        for name, raw_path in optional_dl_paths.items():
            resolved = pathlib.Path(raw_path).resolve()
            if resolved.exists():
                profile_path_dict[name] = resolved
            else:
                log(f"  Skipping {name}: {resolved} does not exist.", log_file)

        log(
            f"{'name':40} | {'shape':>15} | {'nans':>8} | {'infs':>8} | {'dupes':>8} | {'values_above_threshold':>8} | {'all_nan_rows':>12} | {'all_nan_cols':>12} | {'drop?':>6}",
            log_file,
        )
        log(
            f"{'-' * 40} | {'-' * 15} | {'-' * 8} | {'-' * 8} | {'-' * 8} | {'-' * 8} | {'-' * 12} | {'-' * 12} | {'-' * 6}",
            log_file,
        )

        for name, path in profile_path_dict.items():
            df = pd.read_parquet(path)
            nas = df.isna().sum().sum()
            infs = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
            duplicates = df.duplicated().sum()
            values_above_threshold = (
                (df.select_dtypes(include=[np.number]) > threshold).sum().sum()
            )
            shape = df.shape
            # rows/columns that are entirely NaN are candidates to drop
            all_nan_rows = int(
                df[[c for c in df.columns if not c.startswith("Metadata_")]]
                .isna()
                .all(axis=1)
                .sum()
            )
            all_nan_cols = int(df.isna().all(axis=0).sum())
            drop_flag = "YES" if (all_nan_rows or all_nan_cols) else "no"

            log(
                f"{name:40} | {str(shape):>15} | {nas:>8} | {infs:>8} | {duplicates:>8} | {values_above_threshold:>8} | {all_nan_rows:>12} | {all_nan_cols:>12} | {drop_flag:>6}",
                log_file,
            )

print(f"\nLog written to: {log_path}")


Processing patient: NF0014_T1
  Skipping sc_sammed_agg_well: /home/lippincm/mnt/bandicoot/NF1_organoid_data/data/NF0014_T1/image_based_profiles/7.aggregated_profiles/sammed_sc_agg_well_level.parquet does not exist.
  Skipping organoid_sammed_agg_well: /home/lippincm/mnt/bandicoot/NF1_organoid_data/data/NF0014_T1/image_based_profiles/7.aggregated_profiles/sammed_organoid_agg_well_level.parquet does not exist.
  Skipping nucleocentric_sammed_agg_well: /home/lippincm/mnt/bandicoot/NF1_organoid_data/data/NF0014_T1/image_based_profiles/7.aggregated_profiles/sammed_nucleocentric_agg_well_level.parquet does not exist.
  Skipping nucleocentric_morphem_agg_well: /home/lippincm/mnt/bandicoot/NF1_organoid_data/data/NF0014_T1/image_based_profiles/7.aggregated_profiles/nucleocentric_morphem_agg_well_level.parquet does not exist.
  Skipping sc_sammed_agg_treatment: /home/lippincm/mnt/bandicoot/NF1_organoid_data/data/NF0014_T1/image_based_profiles/8.consensus_profiles/sammed_sc_consensus.parquet doe

In [5]:
combined_patient_profiles_path = pathlib.Path(
    f"{profile_base_dir}/data/{output_subdir}/"
).resolve(strict=True)
with open(log_path, "a") as log_file:
    log("= " * 30, log_file)
    log("Processing combined patient profiles", log_file)
    log("= " * 30, log_file)
    # log(f"\n{'=' * 60}", log_file)
    # log(f"Processing patient: {name}", log_file)
    # log(f"{'=' * 60}", log_file)
    log(
        f"{'name':40} | {'shape':>15} | {'nans':>8} | {'infs':>8} | {'dupes':>8} | {'values_above_threshold':>8} | {'all_nan_rows':>12} | {'all_nan_cols':>12} | {'drop?':>6}",
        log_file,
    )
    log(
        f"{'-' * 40} | {'-' * 15} | {'-' * 8} | {'-' * 8} | {'-' * 8} | {'-' * 8} | {'-' * 12} | {'-' * 12} | {'-' * 6}",
        log_file,
    )

    for profile_level in combined_patient_profiles_path.iterdir():
        for profile_file in profile_level.glob("*.parquet"):
            name = profile_file.stem

            df = pd.read_parquet(profile_file)
            nas = df.isna().sum().sum()
            infs = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
            duplicates = df.duplicated().sum()
            values_above_threshold = (
                (df.select_dtypes(include=[np.number]) > threshold).sum().sum()
            )
            shape = df.shape
            # rows/columns that are entirely NaN are candidates to drop
            all_nan_rows = int(
                df[[c for c in df.columns if not c.startswith("Metadata_")]]
                .isna()
                .all(axis=1)
                .sum()
            )
            all_nan_cols = int(df.isna().all(axis=0).sum())
            drop_flag = "YES" if (all_nan_rows or all_nan_cols) else "no"

            log(
                f"{name:40} | {str(shape):>15} | {nas:>8} | {infs:>8} | {duplicates:>8} | {values_above_threshold:>8} | {all_nan_rows:>12} | {all_nan_cols:>12} | {drop_flag:>6}",
                log_file,
            )

= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
Processing combined patient profiles
= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
name                                     |           shape |     nans |     infs |    dupes | values_above_threshold | all_nan_rows | all_nan_cols |  drop?
---------------------------------------- | --------------- | -------- | -------- | -------- | -------- | ------------ | ------------ | ------
organoid_agg_profiles                    |     (680, 1024) |     4847 |        0 |        0 |        0 |            0 |            0 |     no
sc_consensus_profiles                    |     (287, 2815) |       28 |        0 |        0 |        0 |            0 |            0 |     no
organoid_fs_profiles                     |   (73167, 1031) |    61406 |        0 |        0 |        0 |            0 |            0 |     no
sc_profiles                              |   (19649, 2857) |    98377 |        0 |        0 |     9299 |            0